In [1]:
#cell 1
# Install lightweight graph/storage dependencies.

!pip -q install -U torch-geometric tqdm numpy psutil

import sys
import json
import os
import re
import html
import shutil
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import torch
from tqdm.auto import tqdm
from torch_geometric.data import HeteroData

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cpu
CUDA available: False


In [2]:
#cell 2
# Mount Google Drive and define all input/output paths.

from google.colab import drive
drive.mount("/content/drive")

DATASET = "hotpotqa"

PROJECT_DIR = Path("/content/drive/MyDrive/final_project/idea_1")
DATASET_DIR = PROJECT_DIR / "kg" / DATASET
CONSTRUCT_DIR = DATASET_DIR / "kg_construct"
CONSTRUCT_DIR.mkdir(parents=True, exist_ok=True)

# Main chunk file.
CHUNKS_JSON_PATH = PROJECT_DIR / "hotpotqa_docs_chunks.json"

# Inputs already created in previous notebooks.
NORMALIZED_KG_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_kg_extractions.clean.normalized_entities.jsonl"
ENTITY_CATALOG_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_entities.jsonl"

RELATION_ID_TO_ROW_JSONL_PATH = CONSTRUCT_DIR / "relation_id_to_row.jsonl"
FACT_ID_TO_ROW_JSONL_PATH = CONSTRUCT_DIR / "fact_id_to_row.jsonl"

ENTITY_EMBEDDINGS_NPY_PATH = CONSTRUCT_DIR / "hotpotqa_entity_embeddings.float32.npy"
RELATION_EMBEDDINGS_NPY_PATH = CONSTRUCT_DIR / "relation_embeddings.npy"
FACT_EMBEDDINGS_NPY_PATH = CONSTRUCT_DIR / "fact_embeddings.npy"

# New KG outputs.
CHUNK_TEXT_CATALOG_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_chunk_text_catalog.jsonl"
CHUNK_NODES_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_chunk_nodes.jsonl"
CHUNK_ID_TO_NODE_ID_JSON_PATH = CONSTRUCT_DIR / "hotpotqa_chunk_id_to_node_id.json"
CHUNK_NODE_ID_TO_CHUNK_ID_JSON_PATH = CONSTRUCT_DIR / "hotpotqa_chunk_node_id_to_chunk_id.json"

ENTITY_NODES_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_entity_nodes_for_kg.jsonl"

RELATION_EDGES_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_relation_edges_for_kg.jsonl"
FACT_EDGES_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_fact_edges_for_kg.jsonl"
CHUNK_SEQUENCE_EDGES_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_chunk_sequence_edges_for_kg.jsonl"

HETERO_KG_PT_PATH = CONSTRUCT_DIR / "hotpotqa_hetero_kg.pt"
KG_META_JSON_PATH = CONSTRUCT_DIR / "hotpotqa_hetero_kg_meta.json"

required_paths = [
    CHUNKS_JSON_PATH,
    ENTITY_CATALOG_JSONL_PATH,
    RELATION_ID_TO_ROW_JSONL_PATH,
    FACT_ID_TO_ROW_JSONL_PATH,
    ENTITY_EMBEDDINGS_NPY_PATH,
    RELATION_EMBEDDINGS_NPY_PATH,
    FACT_EMBEDDINGS_NPY_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("Project dir:", PROJECT_DIR)
print("Construct dir:", CONSTRUCT_DIR)

Mounted at /content/drive
Project dir: /content/drive/MyDrive/final_project/idea_1
Construct dir: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct


In [3]:
#cell 3
# Define safe writing, JSONL, and normalization helpers.

def atomic_json_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)


def atomic_jsonl_write(records_iterable, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    count = 0
    with open(tmp_path, "w", encoding="utf-8") as f:
        for record in records_iterable:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1

        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)
    return count


def count_jsonl_lines(path):
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
    return count


def normalize_entity_text(text):
    # Match the entity normalization used in previous notebooks.
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u00a0": " ",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip(" \t\r\n\"'`")
    text = text.lower()

    return text


def normalize_title_for_sequence(text):
    # Normalize title only for safe chunk sequence comparison.
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else None


def read_jsonl_row_by_row_number(path, row_number):
    # Read one JSONL row by zero-based row number.
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i == row_number:
                return json.loads(line)

    raise IndexError(f"Row {row_number} not found in {path}")

In [4]:
#cell 4
# Load chunks and create chunk sidecar files.
# Chunk nodes in the graph are lightweight; full text stays in JSONL.

with open(CHUNKS_JSON_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

if not isinstance(chunks, list):
    raise RuntimeError("hotpotqa_docs_chunks.json must be a JSON list.")

num_chunks = len(chunks)

chunk_id_to_node_id = {}
chunk_node_id_to_chunk_id = {}
chunk_id_to_title = {}

chunk_text_records = []
chunk_node_records = []

for chunk_node_id, chunk in enumerate(tqdm(chunks, desc="Preparing chunk nodes")):
    chunk_id = chunk.get("Chunk_id")
    title = chunk.get("Title")
    text = chunk.get("Text")
    paragraph_id = chunk.get("Paragraph_id")
    token_count = chunk.get("Token_count")

    expected_chunk_id = f"hotpotqa_chunk_{chunk_node_id + 1:08d}"

    if chunk_id != expected_chunk_id:
        raise RuntimeError(
            f"Chunk_id order mismatch at node {chunk_node_id}: "
            f"expected {expected_chunk_id}, got {chunk_id}"
        )

    if chunk_id in chunk_id_to_node_id:
        raise RuntimeError(f"Duplicate chunk_id found: {chunk_id}")

    chunk_id_to_node_id[chunk_id] = chunk_node_id
    chunk_node_id_to_chunk_id[str(chunk_node_id)] = chunk_id
    chunk_id_to_title[chunk_id] = title

    chunk_node_records.append({
        "chunk_node_id": chunk_node_id,
        "chunk_id": chunk_id,
    })

    chunk_text_records.append({
        "chunk_node_id": chunk_node_id,
        "chunk_id": chunk_id,
        "title": title,
        "paragraph_id": paragraph_id,
        "token_count": token_count,
        "text": text,
    })

atomic_jsonl_write(chunk_node_records, CHUNK_NODES_JSONL_PATH)
atomic_jsonl_write(chunk_text_records, CHUNK_TEXT_CATALOG_JSONL_PATH)
atomic_json_dump(chunk_id_to_node_id, CHUNK_ID_TO_NODE_ID_JSON_PATH)
atomic_json_dump(chunk_node_id_to_chunk_id, CHUNK_NODE_ID_TO_CHUNK_ID_JSON_PATH)

print("Total chunks:", num_chunks)
print("Saved chunk nodes:", CHUNK_NODES_JSONL_PATH)
print("Saved chunk text catalog:", CHUNK_TEXT_CATALOG_JSONL_PATH)
print("Saved chunk_id -> node_id:", CHUNK_ID_TO_NODE_ID_JSON_PATH)

Preparing chunk nodes:   0%|          | 0/35029 [00:00<?, ?it/s]

Total chunks: 35029
Saved chunk nodes: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_chunk_nodes.jsonl
Saved chunk text catalog: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_chunk_text_catalog.jsonl
Saved chunk_id -> node_id: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_chunk_id_to_node_id.json


In [5]:
#cell 5
# Load unique normalized entities.
# Entity node id must match entity_id used by entity embeddings.

entity_to_id = {}
id_to_entity = {}

entity_node_records = []
bad_entity_rows = []

with open(ENTITY_CATALOG_JSONL_PATH, "r", encoding="utf-8") as f:
    for row, line in enumerate(tqdm(f, desc="Loading entity catalog")):
        if not line.strip():
            continue

        rec = json.loads(line)

        entity_id = rec.get("entity_id")
        entity = rec.get("entity")

        if entity_id != row:
            bad_entity_rows.append({
                "row": row,
                "entity_id": entity_id,
                "entity": entity,
                "reason": "entity_id does not match JSONL row",
            })
            continue

        if not isinstance(entity, str) or not entity:
            bad_entity_rows.append({
                "row": row,
                "entity_id": entity_id,
                "entity": entity,
                "reason": "invalid entity string",
            })
            continue

        if entity in entity_to_id:
            bad_entity_rows.append({
                "row": row,
                "entity_id": entity_id,
                "entity": entity,
                "reason": "duplicate normalized entity",
            })
            continue

        entity_to_id[entity] = entity_id
        id_to_entity[entity_id] = entity

        entity_node_records.append({
            "entity_id": entity_id,
            "entity": entity,
        })

if bad_entity_rows:
    print(json.dumps(bad_entity_rows[:20], ensure_ascii=False, indent=2))
    raise RuntimeError(f"Bad entity catalog rows found: {len(bad_entity_rows)}")

num_entities = len(entity_to_id)

atomic_jsonl_write(entity_node_records, ENTITY_NODES_JSONL_PATH)

print("Total unique entities:", num_entities)
print("Saved entity node catalog:", ENTITY_NODES_JSONL_PATH)
print("First 3 entities:")
print(json.dumps(entity_node_records[:3], ensure_ascii=False, indent=2))

Loading entity catalog: 0it [00:00, ?it/s]

Total unique entities: 327873
Saved entity node catalog: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_nodes_for_kg.jsonl
First 3 entities:
[
  {
    "entity_id": 0,
    "entity": "meet corliss archer"
  },
  {
    "entity_id": 1,
    "entity": "january 7, 1943"
  },
  {
    "entity_id": 2,
    "entity": "september 30, 1956"
  }
]


In [6]:
#cell 6
# Build relation edges from normalized heads/tails.
# PyG graph stores IDs; relation text is stored in relation edge JSONL.

relation_src_entity_ids = []
relation_dst_entity_ids = []
relation_ids = []
relation_chunk_node_ids = []

missing_relation_entities = []
missing_relation_chunks = []

def iter_relation_edge_records():
    with open(RELATION_ID_TO_ROW_JSONL_PATH, "r", encoding="utf-8") as f:
        for row, line in enumerate(tqdm(f, desc="Building relation edges")):
            if not line.strip():
                continue

            rec = json.loads(line)

            relation_id = rec.get("relation_id")
            head = rec.get("head")
            tail = rec.get("tail")
            relation_text = rec.get("relation")
            chunk_id = rec.get("chunk_id")

            if relation_id != row:
                raise RuntimeError(
                    f"relation_id mismatch at row {row}: got {relation_id}"
                )

            if head not in entity_to_id:
                missing_relation_entities.append({
                    "row": row,
                    "relation_id": relation_id,
                    "missing_entity": head,
                    "field": "head",
                    "chunk_id": chunk_id,
                })
                continue

            if tail not in entity_to_id:
                missing_relation_entities.append({
                    "row": row,
                    "relation_id": relation_id,
                    "missing_entity": tail,
                    "field": "tail",
                    "chunk_id": chunk_id,
                })
                continue

            if chunk_id not in chunk_id_to_node_id:
                missing_relation_chunks.append({
                    "row": row,
                    "relation_id": relation_id,
                    "chunk_id": chunk_id,
                })
                continue

            src_id = entity_to_id[head]
            dst_id = entity_to_id[tail]
            chunk_node_id = chunk_id_to_node_id[chunk_id]

            relation_src_entity_ids.append(src_id)
            relation_dst_entity_ids.append(dst_id)
            relation_ids.append(relation_id)
            relation_chunk_node_ids.append(chunk_node_id)

            yield {
                "edge_id": row,
                "edge_type": "relation",
                "source_entity_id": src_id,
                "target_entity_id": dst_id,
                "head": head,
                "tail": tail,
                "relation_id": relation_id,
                "relation": relation_text,
                "chunk_id": chunk_id,
                "chunk_node_id": chunk_node_id,
            }

num_relation_edges_written = atomic_jsonl_write(
    iter_relation_edge_records(),
    RELATION_EDGES_JSONL_PATH,
)

if missing_relation_entities or missing_relation_chunks:
    print("Missing relation entities:", len(missing_relation_entities))
    print(json.dumps(missing_relation_entities[:20], ensure_ascii=False, indent=2))
    print("Missing relation chunks:", len(missing_relation_chunks))
    print(json.dumps(missing_relation_chunks[:20], ensure_ascii=False, indent=2))
    raise RuntimeError("Relation edge construction found missing entities or chunks.")

print("Relation edges:", len(relation_ids))
print("Saved relation edge catalog:", RELATION_EDGES_JSONL_PATH)

Building relation edges: 0it [00:00, ?it/s]

Relation edges: 692402
Saved relation edge catalog: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_relation_edges_for_kg.jsonl


In [7]:
#cell 7
# Build fact edges from entity to chunk.
# PyG graph stores fact_id; fact text is stored in fact edge JSONL.

fact_src_entity_ids = []
fact_dst_chunk_node_ids = []
fact_ids = []

missing_fact_entities = []
missing_fact_chunks = []

def iter_fact_edge_records():
    with open(FACT_ID_TO_ROW_JSONL_PATH, "r", encoding="utf-8") as f:
        for row, line in enumerate(tqdm(f, desc="Building fact edges")):
            if not line.strip():
                continue

            rec = json.loads(line)

            fact_id = rec.get("fact_id")
            entity = rec.get("entity")
            info_text = rec.get("info")
            chunk_id = rec.get("chunk_id")

            if fact_id != row:
                raise RuntimeError(
                    f"fact_id mismatch at row {row}: got {fact_id}"
                )

            if entity not in entity_to_id:
                missing_fact_entities.append({
                    "row": row,
                    "fact_id": fact_id,
                    "missing_entity": entity,
                    "chunk_id": chunk_id,
                })
                continue

            if chunk_id not in chunk_id_to_node_id:
                missing_fact_chunks.append({
                    "row": row,
                    "fact_id": fact_id,
                    "chunk_id": chunk_id,
                })
                continue

            src_id = entity_to_id[entity]
            dst_id = chunk_id_to_node_id[chunk_id]

            fact_src_entity_ids.append(src_id)
            fact_dst_chunk_node_ids.append(dst_id)
            fact_ids.append(fact_id)

            yield {
                "edge_id": row,
                "edge_type": "fact",
                "source_entity_id": src_id,
                "target_chunk_node_id": dst_id,
                "entity": entity,
                "fact_id": fact_id,
                "info": info_text,
                "chunk_id": chunk_id,
            }

num_fact_edges_written = atomic_jsonl_write(
    iter_fact_edge_records(),
    FACT_EDGES_JSONL_PATH,
)

if missing_fact_entities or missing_fact_chunks:
    print("Missing fact entities:", len(missing_fact_entities))
    print(json.dumps(missing_fact_entities[:20], ensure_ascii=False, indent=2))
    print("Missing fact chunks:", len(missing_fact_chunks))
    print(json.dumps(missing_fact_chunks[:20], ensure_ascii=False, indent=2))
    raise RuntimeError("Fact edge construction found missing entities or chunks.")

print("Fact edges:", len(fact_ids))
print("Saved fact edge catalog:", FACT_EDGES_JSONL_PATH)

Building fact edges: 0it [00:00, ?it/s]

Fact edges: 559959
Saved fact edge catalog: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_fact_edges_for_kg.jsonl


In [8]:
#cell 8
# Build directed chunk sequence edges only inside the same Wikipedia title.

next_src_chunk_node_ids = []
next_dst_chunk_node_ids = []

prev_src_chunk_node_ids = []
prev_dst_chunk_node_ids = []

sequence_records = []

for i in tqdm(range(num_chunks - 1), desc="Building chunk sequence edges"):
    current_chunk = chunks[i]
    next_chunk = chunks[i + 1]

    current_title = normalize_title_for_sequence(current_chunk.get("Title"))
    next_title = normalize_title_for_sequence(next_chunk.get("Title"))

    if current_title != next_title:
        continue

    current_chunk_id = current_chunk["Chunk_id"]
    next_chunk_id = next_chunk["Chunk_id"]

    current_node_id = chunk_id_to_node_id[current_chunk_id]
    next_node_id = chunk_id_to_node_id[next_chunk_id]

    # next_chunk: current -> next
    next_src_chunk_node_ids.append(current_node_id)
    next_dst_chunk_node_ids.append(next_node_id)

    sequence_records.append({
        "edge_type": "next_chunk",
        "source_chunk_node_id": current_node_id,
        "target_chunk_node_id": next_node_id,
        "source_chunk_id": current_chunk_id,
        "target_chunk_id": next_chunk_id,
        "title": current_chunk.get("Title"),
    })

    # prev_chunk: next -> current
    prev_src_chunk_node_ids.append(next_node_id)
    prev_dst_chunk_node_ids.append(current_node_id)

    sequence_records.append({
        "edge_type": "prev_chunk",
        "source_chunk_node_id": next_node_id,
        "target_chunk_node_id": current_node_id,
        "source_chunk_id": next_chunk_id,
        "target_chunk_id": current_chunk_id,
        "title": current_chunk.get("Title"),
    })

atomic_jsonl_write(sequence_records, CHUNK_SEQUENCE_EDGES_JSONL_PATH)

print("next_chunk edges:", len(next_src_chunk_node_ids))
print("prev_chunk edges:", len(prev_src_chunk_node_ids))
print("Saved chunk sequence edge catalog:", CHUNK_SEQUENCE_EDGES_JSONL_PATH)

Building chunk sequence edges:   0%|          | 0/35028 [00:00<?, ?it/s]

next_chunk edges: 25219
prev_chunk edges: 25219
Saved chunk sequence edge catalog: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_chunk_sequence_edges_for_kg.jsonl


In [9]:
#cell 9
# Validate that embedding rows match graph node/edge IDs.

entity_embeddings = np.load(ENTITY_EMBEDDINGS_NPY_PATH, mmap_mode="r")
relation_embeddings = np.load(RELATION_EMBEDDINGS_NPY_PATH, mmap_mode="r")
fact_embeddings = np.load(FACT_EMBEDDINGS_NPY_PATH, mmap_mode="r")

print("Entity embeddings shape:", entity_embeddings.shape)
print("Relation embeddings shape:", relation_embeddings.shape)
print("Fact embeddings shape:", fact_embeddings.shape)

if entity_embeddings.shape[0] != num_entities:
    raise RuntimeError(
        f"Entity embedding row mismatch: {entity_embeddings.shape[0]} != {num_entities}"
    )

if relation_embeddings.shape[0] != len(relation_ids):
    raise RuntimeError(
        f"Relation embedding row mismatch: {relation_embeddings.shape[0]} != {len(relation_ids)}"
    )

if fact_embeddings.shape[0] != len(fact_ids):
    raise RuntimeError(
        f"Fact embedding row mismatch: {fact_embeddings.shape[0]} != {len(fact_ids)}"
    )

if entity_embeddings.dtype != np.float32:
    raise RuntimeError(f"Entity embeddings must be float32, got {entity_embeddings.dtype}")

if relation_embeddings.dtype != np.float32:
    raise RuntimeError(f"Relation embeddings must be float32, got {relation_embeddings.dtype}")

if fact_embeddings.dtype != np.float32:
    raise RuntimeError(f"Fact embeddings must be float32, got {fact_embeddings.dtype}")

embedding_dim = int(entity_embeddings.shape[1])

if relation_embeddings.shape[1] != embedding_dim:
    raise RuntimeError("Relation embedding dim does not match entity embedding dim.")

if fact_embeddings.shape[1] != embedding_dim:
    raise RuntimeError("Fact embedding dim does not match entity embedding dim.")

print("Embedding validation passed.")
print("Embedding dim:", embedding_dim)

Entity embeddings shape: (327873, 4096)
Relation embeddings shape: (692402, 4096)
Fact embeddings shape: (559959, 4096)
Embedding validation passed.
Embedding dim: 4096


In [10]:
# cell 10
# Build the heterogeneous knowledge graph as PyG HeteroData.

data = HeteroData()

# Node types.
data["entity"].num_nodes = num_entities
data["entity"].entity_id = torch.arange(num_entities, dtype=torch.long)

data["chunk"].num_nodes = num_chunks
data["chunk"].chunk_node_id = torch.arange(num_chunks, dtype=torch.long)

# Edge type 1: entity --relation--> entity.
data[("entity", "relation", "entity")].edge_index = torch.tensor(
    [relation_src_entity_ids, relation_dst_entity_ids],
    dtype=torch.long,
)

data[("entity", "relation", "entity")].relation_id = torch.tensor(
    relation_ids,
    dtype=torch.long,
)

data[("entity", "relation", "entity")].chunk_node_id = torch.tensor(
    relation_chunk_node_ids,
    dtype=torch.long,
)

# Edge type 2: entity --fact--> chunk.
data[("entity", "fact", "chunk")].edge_index = torch.tensor(
    [fact_src_entity_ids, fact_dst_chunk_node_ids],
    dtype=torch.long,
)

data[("entity", "fact", "chunk")].fact_id = torch.tensor(
    fact_ids,
    dtype=torch.long,
)

# Edge type 3a: chunk --next_chunk--> chunk.
data[("chunk", "next_chunk", "chunk")].edge_index = torch.tensor(
    [next_src_chunk_node_ids, next_dst_chunk_node_ids],
    dtype=torch.long,
)

# Edge type 3b: chunk --prev_chunk--> chunk.
data[("chunk", "prev_chunk", "chunk")].edge_index = torch.tensor(
    [prev_src_chunk_node_ids, prev_dst_chunk_node_ids],
    dtype=torch.long,
)

# Save graph.
torch.save(data, HETERO_KG_PT_PATH)

print(data)
print("Saved PyG HeteroData KG:", HETERO_KG_PT_PATH)
print("File size MB:", round(file_size_mb(HETERO_KG_PT_PATH), 2))

HeteroData(
  entity={
    num_nodes=327873,
    entity_id=[327873],
  },
  chunk={
    num_nodes=35029,
    chunk_node_id=[35029],
  },
  (entity, relation, entity)={
    edge_index=[2, 692402],
    relation_id=[692402],
    chunk_node_id=[692402],
  },
  (entity, fact, chunk)={
    edge_index=[2, 559959],
    fact_id=[559959],
  },
  (chunk, next_chunk, chunk)={ edge_index=[2, 25219] },
  (chunk, prev_chunk, chunk)={ edge_index=[2, 25219] }
)
Saved PyG HeteroData KG: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_hetero_kg.pt
File size MB: 37.49


In [11]:
# cell 11
# Save a metadata file that documents all node/edge types and sidecar files.

kg_meta = {
    "dataset": DATASET,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "graph_format": "torch_geometric.data.HeteroData",
    "graph_path": str(HETERO_KG_PT_PATH),
    "node_types": {
        "entity": {
            "num_nodes": num_entities,
            "id_field": "entity_id",
            "text_source": str(ENTITY_NODES_JSONL_PATH),
            "embedding_source": str(ENTITY_EMBEDDINGS_NPY_PATH),
        },
        "chunk": {
            "num_nodes": num_chunks,
            "id_field": "chunk_node_id",
            "chunk_id_source": str(CHUNK_NODES_JSONL_PATH),
            "text_source": str(CHUNK_TEXT_CATALOG_JSONL_PATH),
        },
    },
    "edge_types": {
        "entity__relation__entity": {
            "pyg_edge_type": ["entity", "relation", "entity"],
            "num_edges": len(relation_ids),
            "directed": True,
            "edge_id_field": "relation_id",
            "edge_text_source": str(RELATION_EDGES_JSONL_PATH),
            "embedding_source": str(RELATION_EMBEDDINGS_NPY_PATH),
            "extra_tensor_fields": ["relation_id", "chunk_node_id"],
        },
        "entity__fact__chunk": {
            "pyg_edge_type": ["entity", "fact", "chunk"],
            "num_edges": len(fact_ids),
            "directed": True,
            "edge_id_field": "fact_id",
            "edge_text_source": str(FACT_EDGES_JSONL_PATH),
            "embedding_source": str(FACT_EMBEDDINGS_NPY_PATH),
            "extra_tensor_fields": ["fact_id"],
        },
        "chunk__next_chunk__chunk": {
            "pyg_edge_type": ["chunk", "next_chunk", "chunk"],
            "num_edges": len(next_src_chunk_node_ids),
            "directed": True,
            "edge_text_source": str(CHUNK_SEQUENCE_EDGES_JSONL_PATH),
        },
        "chunk__prev_chunk__chunk": {
            "pyg_edge_type": ["chunk", "prev_chunk", "chunk"],
            "num_edges": len(prev_src_chunk_node_ids),
            "directed": True,
            "edge_text_source": str(CHUNK_SEQUENCE_EDGES_JSONL_PATH),
        },
    },
    "sidecar_files": {
        "chunk_text_catalog_jsonl": str(CHUNK_TEXT_CATALOG_JSONL_PATH),
        "chunk_nodes_jsonl": str(CHUNK_NODES_JSONL_PATH),
        "chunk_id_to_node_id_json": str(CHUNK_ID_TO_NODE_ID_JSON_PATH),
        "chunk_node_id_to_chunk_id_json": str(CHUNK_NODE_ID_TO_CHUNK_ID_JSON_PATH),
        "entity_nodes_jsonl": str(ENTITY_NODES_JSONL_PATH),
        "relation_edges_jsonl": str(RELATION_EDGES_JSONL_PATH),
        "fact_edges_jsonl": str(FACT_EDGES_JSONL_PATH),
        "chunk_sequence_edges_jsonl": str(CHUNK_SEQUENCE_EDGES_JSONL_PATH),
    },
    "embedding_files": {
        "entity_embeddings_npy": str(ENTITY_EMBEDDINGS_NPY_PATH),
        "relation_embeddings_npy": str(RELATION_EMBEDDINGS_NPY_PATH),
        "fact_embeddings_npy": str(FACT_EMBEDDINGS_NPY_PATH),
        "embedding_dim": embedding_dim,
        "dtype": "float32",
    },
    "design_note": (
        "String text is intentionally stored in JSONL sidecar files. "
        "The PyG graph stores only integer node IDs, edge IDs, and edge_index tensors."
    ),
}

atomic_json_dump(kg_meta, KG_META_JSON_PATH)

print("Saved KG metadata:", KG_META_JSON_PATH)
print(json.dumps(kg_meta, ensure_ascii=False, indent=2)[:3000])

Saved KG metadata: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_hetero_kg_meta.json
{
  "dataset": "hotpotqa",
  "created_at_utc": "2026-05-23T07:39:22.009439+00:00",
  "graph_format": "torch_geometric.data.HeteroData",
  "graph_path": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_hetero_kg.pt",
  "node_types": {
    "entity": {
      "num_nodes": 327873,
      "id_field": "entity_id",
      "text_source": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_nodes_for_kg.jsonl",
      "embedding_source": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_embeddings.float32.npy"
    },
    "chunk": {
      "num_nodes": 35029,
      "id_field": "chunk_node_id",
      "chunk_id_source": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_chunk_nodes.jsonl",
      "text_source": "/content/drive/MyDrive/final_project/idea_1/kg/hotpo

In [12]:
# cell 12
# Reload and verify the saved KG.

loaded_data = torch.load(HETERO_KG_PT_PATH, map_location="cpu", weights_only=False)

print(loaded_data)

assert loaded_data["entity"].num_nodes == num_entities
assert loaded_data["chunk"].num_nodes == num_chunks

assert loaded_data[("entity", "relation", "entity")].edge_index.shape[1] == len(relation_ids)
assert loaded_data[("entity", "fact", "chunk")].edge_index.shape[1] == len(fact_ids)
assert loaded_data[("chunk", "next_chunk", "chunk")].edge_index.shape[1] == len(next_src_chunk_node_ids)
assert loaded_data[("chunk", "prev_chunk", "chunk")].edge_index.shape[1] == len(prev_src_chunk_node_ids)

print("Graph reload verification passed.")

print("\nOutput files:")
output_paths = [
    HETERO_KG_PT_PATH,
    KG_META_JSON_PATH,
    CHUNK_TEXT_CATALOG_JSONL_PATH,
    CHUNK_NODES_JSONL_PATH,
    ENTITY_NODES_JSONL_PATH,
    RELATION_EDGES_JSONL_PATH,
    FACT_EDGES_JSONL_PATH,
    CHUNK_SEQUENCE_EDGES_JSONL_PATH,
]

for path in output_paths:
    print(f"{path.name}: {round(file_size_mb(path), 2)} MB")

# Inspect one relation edge.
sample_relation_edge = read_jsonl_row_by_row_number(RELATION_EDGES_JSONL_PATH, 0)
print("\nSample relation edge:")
print(json.dumps(sample_relation_edge, ensure_ascii=False, indent=2))

# Inspect one fact edge.
sample_fact_edge = read_jsonl_row_by_row_number(FACT_EDGES_JSONL_PATH, 0)
print("\nSample fact edge:")
print(json.dumps(sample_fact_edge, ensure_ascii=False, indent=2))

# Inspect one chunk text record.
sample_chunk_text = read_jsonl_row_by_row_number(CHUNK_TEXT_CATALOG_JSONL_PATH, 0)
print("\nSample chunk text record:")
print(json.dumps(sample_chunk_text, ensure_ascii=False, indent=2)[:1500])

HeteroData(
  entity={
    num_nodes=327873,
    entity_id=[327873],
  },
  chunk={
    num_nodes=35029,
    chunk_node_id=[35029],
  },
  (entity, relation, entity)={
    edge_index=[2, 692402],
    relation_id=[692402],
    chunk_node_id=[692402],
  },
  (entity, fact, chunk)={
    edge_index=[2, 559959],
    fact_id=[559959],
  },
  (chunk, next_chunk, chunk)={ edge_index=[2, 25219] },
  (chunk, prev_chunk, chunk)={ edge_index=[2, 25219] }
)
Graph reload verification passed.

Output files:
hotpotqa_hetero_kg.pt: 37.49 MB
hotpotqa_hetero_kg_meta.json: 0.0 MB
hotpotqa_chunk_text_catalog.jsonl: 62.69 MB
hotpotqa_chunk_nodes.jsonl: 2.13 MB
hotpotqa_entity_nodes_for_kg.jsonl: 16.14 MB
hotpotqa_relation_edges_for_kg.jsonl: 207.74 MB
hotpotqa_fact_edges_for_kg.jsonl: 151.1 MB
hotpotqa_chunk_sequence_edges_for_kg.jsonl: 10.24 MB

Sample relation edge:
{
  "edge_id": 0,
  "edge_type": "relation",
  "source_entity_id": 0,
  "target_entity_id": 1,
  "head": "meet corliss archer",
  "tail": "ja

In [13]:
#cell 13
# Define helper functions for future traversal notebooks.

def load_kg_package():
    # Load graph and lightweight maps.
    kg = torch.load(HETERO_KG_PT_PATH, map_location="cpu", weights_only=False)

    with open(CHUNK_ID_TO_NODE_ID_JSON_PATH, "r", encoding="utf-8") as f:
        chunk_id_to_node = json.load(f)

    with open(CHUNK_NODE_ID_TO_CHUNK_ID_JSON_PATH, "r", encoding="utf-8") as f:
        chunk_node_to_id = json.load(f)

    return kg, chunk_id_to_node, chunk_node_to_id


def get_chunk_text_by_node_id(chunk_node_id):
    # Read one chunk text by node id.
    return read_jsonl_row_by_row_number(CHUNK_TEXT_CATALOG_JSONL_PATH, int(chunk_node_id))


def get_relation_edge_by_relation_id(relation_id):
    # relation_id is equal to row number.
    return read_jsonl_row_by_row_number(RELATION_EDGES_JSONL_PATH, int(relation_id))


def get_fact_edge_by_fact_id(fact_id):
    # fact_id is equal to row number.
    return read_jsonl_row_by_row_number(FACT_EDGES_JSONL_PATH, int(fact_id))


kg, chunk_id_to_node, chunk_node_to_id = load_kg_package()

print("Loaded KG package.")
print("Node types:", kg.node_types)
print("Edge types:", kg.edge_types)

# Example: inspect outgoing relation edges for one entity.
entity_name = "german language"
entity_id = entity_to_id.get(entity_name)

if entity_id is not None:
    rel_edge_index = kg[("entity", "relation", "entity")].edge_index
    rel_ids_tensor = kg[("entity", "relation", "entity")].relation_id

    outgoing_positions = (rel_edge_index[0] == entity_id).nonzero(as_tuple=False).view(-1)

    print(f"\nOutgoing relation edges for entity={entity_name!r}: {len(outgoing_positions)}")

    for pos in outgoing_positions[:5].tolist():
        relation_id = int(rel_ids_tensor[pos])
        edge_rec = get_relation_edge_by_relation_id(relation_id)
        print(json.dumps(edge_rec, ensure_ascii=False, indent=2))
else:
    print(f"Entity not found in KG: {entity_name}")

Loaded KG package.
Node types: ['entity', 'chunk']
Edge types: [('entity', 'relation', 'entity'), ('entity', 'fact', 'chunk'), ('chunk', 'next_chunk', 'chunk'), ('chunk', 'prev_chunk', 'chunk')]

Outgoing relation edges for entity='german language': 58
{
  "edge_id": 109081,
  "edge_type": "relation",
  "source_entity_id": 63882,
  "target_entity_id": 63859,
  "head": "german language",
  "tail": "high german consonant shift",
  "relation_id": 109081,
  "relation": "The history of the German language begins with the High German consonant shift.",
  "chunk_id": "hotpotqa_chunk_00005439",
  "chunk_node_id": 5438
}
{
  "edge_id": 109130,
  "edge_type": "relation",
  "source_entity_id": 63882,
  "target_entity_id": 63905,
  "head": "german language",
  "tail": "middle high german",
  "relation_id": 109130,
  "relation": "German language underwent significant linguistic changes in syntax, phonetics, and morphology during the Middle High German period.",
  "chunk_id": "hotpotqa_chunk_0000544